In [10]:
# 앙상블 리트리버 설정
# --> 두 개의 리트리버를 결합하여 하이브리드 검색을 수행한다.
# ensemble_retriever = EnsembleRetriever(
#     retrievers=[bm25_retriever, chroma_retriever],
#     weights=[0.2, 0.8]
# )

In [11]:
# 라이브러리 불러오기
import bs4
from langchain import hub
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.prompts import ChatPromptTemplate
from langchain.retrievers import BM25Retriever, EnsembleRetriever
from dotenv import load_dotenv

load_dotenv()

True

In [12]:
# 문서 로드
loader = WebBaseLoader(
    web_paths=("https://news.naver.com/section/101",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("sa_text", "sa_item_SECTION_HEADLINE")
        )
    ),
)
docs = loader.load()

# 텍스트 분할
# .from_tiktoken_encoder() --> OpenAI의 tiktoken 토크나이저를 활용하여 텍스트를 
#       토큰 단위로 분할한다. 일반적인 문자 단위 분할보다 토큰 단위 분할은 
#       언어 모델의 토큰화 방식과 일치하여 더 정확한 청크 분할이 가능하다.
#       각 청크의 토큰 수를 정확하게 제어할 수 있다.
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300,
    chunk_overlap=50
)

splits = text_splitter.split_documents(docs)

In [15]:
from openai import OpenAI
# 벡터 스토어
vectorstore = Chroma.from_documents(
    documents=splits, 
    embedding=OpenAIEmbeddings()
)
# poetry add chromadb
# 크로마 리트리버 생성
chroma_retriever = vectorstore.as_retriever(
    search_type="mmr", # MMR 알고리즘을 사용하여 검색
    search_kwargs={'k':1,'fetch_k':4} # 상위 1개의 문서를 반환하지만, 고려할 문서는 4개로 설정
)

# poetry add rank_bm25
# BM25 리트리버 설정
bm25_retriever = BM25Retriever.from_documents(splits)
bm25_retriever.k = 3  # Retrieve top 2 results --> 질문과 가장 관련있는 문서 3개

# 앙상블 리트리버 설정
# --> 두 개의 리트리버를 결합하여 하이브리드 검색을 수행한다.
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, chroma_retriever],
    weights=[0.2, 0.8]
)

In [16]:
# ---------------

template = """다음 맥락을 바탕으로 질문에 답변하세요:

{context}

질문: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

# LLM
llm = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)

# Post-processing
def format_docs(docs):
    formatted = "\n\n".join(doc.page_content for doc in docs)
    return formatted

# Chain
rag_chain = (
    {"context": ensemble_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Question
rag_chain.invoke("상반기 경제에 대해서 알려줘")

'상반기 경제에 대한 정보는 여러 기사에서 다루어지고 있습니다. 주요 내용은 다음과 같습니다:\n\n1. **기업 실적**: 올해 상반기 코스피 상장사들은 총 110조 원대의 영업이익을 기록하며 전년 동기 대비 증가했습니다. 그러나 일부 기업들은 영업이익이 감소하는 모습을 보였습니다. 예를 들어, BNK금융의 영업이익은 24% 감소했고, 화승엔터프라이즈도 6% 줄어들었습니다.\n\n2. **산업별 동향**: 방산주가 상반기 동안 주도주 역할을 했으나, 최근에는 주가 조정 국면에 접어들었습니다. 이는 러시아-우크라이나 전쟁의 종식 가능성이 제기되면서 주가 고평가 논란이 일어난 것과 관련이 있습니다.\n\n3. **중소기업 우려**: 더불어민주당의 노조법 개정안(노란봉투법)에 대한 중소기업계의 우려가 커지고 있습니다. 중소기업은 대기업에 비해 구조가 취약하기 때문에, 법 개정이 경제의 근간에 부정적인 영향을 미칠 수 있다는 주장이 제기되고 있습니다.\n\n이러한 요소들은 상반기 경제의 전반적인 흐름과 기업들의 실적에 중요한 영향을 미치고 있습니다.'